# 🧠 Kairos CIDA v4: Latent Policy RL Training (Teacher-in-the-Loop)

This notebook implements **Latent Policy Reinforcement Learning (LPRL)**.

### 🎯 The Philosophy:
CIDA is a **blind agent** operating inside the hidden layers of the model. It does not know the answer key. Its task is to perform latent moves (shifting vectors via $\delta$) and learn **how to search and resolve contradictions** (action policies).

The **Teacher (7B Model)** acts as a frozen critic/environment. Guided by a rich instruction prompt, the Teacher evaluates the *reasoning process* generated by the student, scoring it on logic, consistency, and constraint satisfaction.

We optimize CIDA using **Policy Gradient (REINFORCE)** based on the Teacher's logical rewards. This prevents memorization and forces CIDA to learn generalized *cognitive navigation strategies* (how to walk in the gradient space).

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes huggingface_hub tqdm

import getpass, sys, os
token = getpass.getpass("GitHub PAT: ").strip()
if token:
    !git clone https://{token}@github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin
else:
    !git clone https://github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from cida_plugin.config import CIDAPluginConfig
from cida_plugin.core import CIDAPlugin
from cida_plugin.projector import LatentAEProjector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STUDENT_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TEACHER_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("Loading Tokenizers...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME, padding_side="left")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

print("Loading Student LLM (1.5B)...")
student_llm = AutoModelForCausalLM.from_pretrained(STUDENT_NAME, quantization_config=bnb, device_map="auto")
student_llm.eval()

print("Loading Teacher LLM (7B)...")
teacher_llm = AutoModelForCausalLM.from_pretrained(TEACHER_NAME, quantization_config=bnb, device_map="auto")
teacher_llm.eval()

# Initialize CIDA config
cfg = CIDAPluginConfig(
    d_input=student_llm.config.hidden_size, d_hidden=1024, d_message=1024,
    num_agents=4, num_classes=2, max_rounds=4,
    num_attn_heads=8, use_perspective_projector=True,
    anonymous_messages=True, sparse_comm_k=3, use_dynamic_trust=True
)
cida = CIDAPlugin(cfg).to(DEVICE).float()
proj = LatentAEProjector(cfg.d_hidden, student_llm.config.hidden_size, cfg.num_agents, num_plan=8).to(DEVICE)

# Locate CIDA pre-trained SFT weights (either local or drive)
drive_path = "/content/drive/MyDrive/cida_v4_universal_weights"
local_path = "cida_v4_universal_weights"
v4_dir = drive_path if os.path.exists(os.path.join(drive_path, "pytorch_model.bin")) else (local_path if os.path.exists(os.path.join(local_path, "pytorch_model.bin")) else None)

if v4_dir:
    print(f"Loading pre-trained CIDA weights from: {v4_dir} (Warm start)")
    try:
        cida.load_state_dict(torch.load(os.path.join(v4_dir, "pytorch_model.bin"), map_location="cpu"))
        proj.load_state_dict(torch.load(os.path.join(v4_dir, "projector.pt"), map_location="cpu"))
        print("✅ Pre-trained CIDA weights loaded successfully! Ready for RL alignment.")
    except Exception as e:
        print(f"Failed to load weights: {e}. Falling back to training from scratch.")
else:
    print("ℹ️ Pre-trained weights not found. Initializing fresh CIDA from scratch.")

In [ ]:
import torch.optim as optim
import json, re

# 1. Teacher Critique Prompt
TEACHER_SYSTEM_PROMPT = (
    "You are an expert logic critic. Evaluate the Student's reasoning path.\n"
    "Analyze if the student satisfied constraints, avoided local biases, and reasoned correctly.\n"
    "Provide your evaluation in JSON format with exactly two fields:\n"
    "1. 'score': a float between 0.0 (illogical) and 1.0 (perfectly correct).\n"
    "2. 'hint': a short, 1-sentence hint/clue pointing to the correct logic (e.g. 'Focus on Friday departure constraint').\n"
    "Output ONLY the raw JSON block, nothing else."
)

def make_teacher_prompt(q, student_response, correct_answer):
    return (
        f"<|im_start|>system\n{TEACHER_SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Question: {q}\n"
        f"Correct Option Key: {correct_answer}\n"
        f"Student's Reasoning & Output: {student_response}\n\n"
        f"Evaluate the logical coherence. Output JSON: "
        f"<|im_end|>\n<|im_start|>assistant\n"
    )

def evaluate_with_teacher(question, student_response, correct_answer):
    t_prompt = make_teacher_prompt(question, student_response, correct_answer)
    t_enc = tokenizer(t_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        t_ids = teacher_llm.generate(**t_enc, max_new_tokens=60, do_sample=False)
        t_ans = tokenizer.decode(t_ids[0, t_enc.input_ids.size(1):], skip_special_tokens=True).strip()

    # Parse JSON output
    try:
        # Find JSON block using regex if any extra characters were output
        match = re.search(r'\{.*\}', t_ans, re.DOTALL)
        data = json.loads(match.group(0) if match else t_ans)
        score = min(max(float(data.get("score", 0.0)), 0.0), 1.0)
        hint = data.get("hint", "Focus on the logical constraints of the question.").strip()
        return score, hint
    except:
        # Fallback
        score = 0.5 if correct_answer in student_response else 0.0
        hint = "Focus on the logical constraints of the question."
        return score, hint

# 2. RL Optimizer
optimizer = optim.Adam(list(cida.parameters()) + list(proj.parameters()), lr=1e-4)

def train_rl_step(sample):
    q = sample["question"]
    correct_ans = sample["answer"]

    prompt = f"<|im_start|>system\nSolve logically.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Student Forward Pass
    with torch.no_grad():
        out = student_llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
        h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
        p_emb = student_llm.get_input_embeddings()(enc.input_ids).detach()

    # CIDA agent action selection (continuous latent shift)
    co = cida(pooled=h_base)
    plan = proj(co["state_final"].s) # shape: (1, 8, d_llm)

    disagreement = co["disagreement"].mean()

    # Generate answer from student model
    with torch.no_grad():
        cat = torch.cat([p_emb, plan.to(p_emb.dtype)], dim=1).to(p_emb.dtype)
        pm = torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)
        gm = torch.cat([enc.attention_mask, pm], dim=1)
        ids = student_llm.generate(inputs_embeds=cat, attention_mask=gm, max_new_tokens=100, do_sample=True, temperature=0.7)
        student_ans = tokenizer.decode(ids[0], skip_special_tokens=True).strip()

    # Get score and verbal hint from SOTA Teacher Model
    reward, hint_text = evaluate_with_teacher(q, student_ans, correct_ans)

    # Get embedding of the teacher's hint under the student's embedding layer
    # This gives a dense target direction for CIDA's plan vector!
    hint_tokens = tokenizer(hint_text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        hint_emb = student_llm.get_input_embeddings()(hint_tokens).mean(dim=1).float() # shape: (1, d_llm)

    # Compute Critique Alignment Loss: maximize cosine similarity between plan tokens and hint embedding
    plan_mean = plan.mean(dim=1).float() # shape: (1, d_llm)
    cosine_sim = torch.nn.functional.cosine_similarity(plan_mean, hint_emb)
    align_loss = 1.0 - cosine_sim.mean()

    # Optimize: Combined Policy Gradient and Critique Alignment
    # This provides dense, high-dimensional gradients to CIDA to prevent getting stuck!
    optimizer.zero_grad()
    rl_loss = disagreement * (1.0 - reward) + 0.3 * align_loss
    rl_loss.backward()
    optimizer.step()

    return reward, hint_text, rl_loss.item(), student_ans

print("✅ RL Policy Training Engine with Critique Alignment compiled!")

In [ ]:
from datasets import load_dataset
import random

print("Loading OpenThoughts-114k (SOTA reasoning traces)...")
try:
    ot_dataset = load_dataset("open-thoughts/OpenThoughts-114k", "metadata", split="train", streaming=True)
    rl_dataset = []
    for i, item in enumerate(ot_dataset):
        if len(rl_dataset) >= 1000: break
        q = item.get("problem", "")
        cot = item.get("deepseek_reasoning", "")
        ans = item.get("deepseek_solution", "")
        domain = item.get("domain", "")

        # Keep reasoning/logic tasks with verifiable answers from math, logic, puzzles or science domains
        if domain in ["math", "puzzles", "logic", "science"] and len(q) > 30 and len(ans) > 0 and len(cot) > 100:
            rl_dataset.append({
                "question": q,
                "answer": ans
            })
    print(f"✅ Loaded {len(rl_dataset)} SOTA tasks from OpenThoughts-114k!")
except Exception as e:
    print(f"Error loading OpenThoughts-114k: {e}. Falling back to toy dataset.")
    rl_dataset = [
        {
            "question": "AI travel agent booking from Astana to Dubai. Strict constraints: 1. Price < $1000. 2. Depart on Friday. Flights: A: Wednesday $800, B: Friday $950, C: Friday $1200.",
            "answer": "B"
        }
    ]

print("Starting Teacher-Guided RL Policy training on OpenThoughts...")
for epoch in range(3):
    total_reward = 0.0
    # Run on 100 samples per epoch to demonstrate training speedup
    for idx, sample in enumerate(rl_dataset[:100]):
        reward, hint_text, loss, ans = train_rl_step(sample)
        total_reward += reward
        if (idx + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/3 | Step {idx+1}/100 | Avg Reward: {total_reward/(idx+1):.2f} | Loss: {loss:.4f}")
            print(f"  -> Teacher Hint: '{hint_text}'")
            print(f"  -> Sample Student Answer: {ans[:80]}...")
            print("-" * 50)